In [1]:

import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA_DIR = Path(r"F:\FunTrust\WEEK 4")

customers = pd.read_csv(DATA_DIR / "FinTrust_Customer_Data.csv", sep=";", decimal=",")
tx = pd.read_csv(DATA_DIR / "FinTrust_Transaction_Data.csv", sep=";", decimal=",")

tx["Transaction_DateTime"] = pd.to_datetime(
    tx["Transaction_DateTime"], format="%d/%m/%Y %H:%M", errors="coerce"
)

print("Clients      :", customers.shape)
print("Transactions :", tx.shape)
print("Dates invalides :", tx["Transaction_DateTime"].isna().sum())
print("Période         :", tx["Transaction_DateTime"].min(), "->", tx["Transaction_DateTime"].max())
print()
print(tx.dtypes)
print()
print(tx["Transaction_Status"].value_counts(dropna=False))
display(tx.head(3))

Clients      : (1500, 12)
Transactions : (12000, 11)
Dates invalides : 0
Période         : 2026-01-01 00:00:00 -> 2026-03-31 23:59:00

Transaction_ID                          str
Customer_ID                             str
Transaction_DateTime         datetime64[us]
Transaction_Type                        str
Amount_NGN                          float64
Channel                                 str
Device_Type                             str
Location                                str
International_Transaction               str
Transaction_Status                      str
Risk_Review_Flag                        str
dtype: object

Transaction_Status
Successful    10856
Failed          630
Reversed        326
Pending         188
Name: count, dtype: int64


,Transaction_ID,Customer_ID,Transaction_DateTime,Transaction_Type,Amount_NGN,Channel,Device_Type,Location,International_Transaction,Transaction_Status,Risk_Review_Flag
0,FT-T000001,FT-C01135,2026-01-01 00:00:00,Card Purchase,5923.80,Mobile App,Android,Enugu,No,Reversed,Yes
1,FT-T000002,FT-C00428,2026-01-01 00:10:00,Cash Withdrawal,11681.12,ATM,Android,Kaduna,No,Successful,Yes
2,FT-T000003,FT-C00469,2026-01-01 00:21:00,Transfer,571.91,ATM,POS Terminal,Kano,No,Successful,No


In [2]:
CURRENCY = "NGN"

def to_flag(s):
    return s.astype(str).str.strip().str.lower().isin({"yes", "y", "1", "true", "oui"})

def summarize(df, customers, label):
    amount = df["Amount_NGN"]
    return pd.Series({
        "Clients (table clients)": customers["Customer_ID"].nunique(),
        "Clients avec transaction": df["Customer_ID"].nunique(),
        "Transactions": len(df),
        f"Volume total ({CURRENCY})": amount.sum(),
        f"Moyenne ({CURRENCY})": amount.mean(),
        f"Médiane ({CURRENCY})": amount.median(),
        "Taux de signalement revue (%)": to_flag(df["Risk_Review_Flag"]).mean() * 100,
    }, name=label)

# Contrôle qualité
print("Doublons Transaction_ID :", tx["Transaction_ID"].duplicated().sum())
print("Montants manquants      :", tx["Amount_NGN"].isna().sum())
print("Transactions sans client:", (~tx["Customer_ID"].isin(customers["Customer_ID"])).sum())
print()

# Toutes les transactions vs réussies uniquement
result = pd.concat([
    summarize(tx, customers, "Toutes"),
    summarize(tx[tx["Transaction_Status"] == "Successful"], customers, "Réussies"),
], axis=1)
display(result.round(2))

# Détail par statut
by_status = tx.groupby("Transaction_Status").agg(
    transactions=("Transaction_ID", "count"),
    volume=("Amount_NGN", "sum"),
    montant_moyen=("Amount_NGN", "mean"),
    taux_signalement=("Risk_Review_Flag", lambda s: to_flag(s).mean() * 100),
).round(2)
display(by_status)

Doublons Transaction_ID : 0
Montants manquants      : 0
Transactions sans client: 0



,Toutes,Réussies
Clients (table clients),1.500000e+03,1.500000e+03
Clients avec transaction,1.500000e+03,1.500000e+03
Transactions,1.200000e+04,1.085600e+04
Volume total (NGN),5.604774e+08,5.108074e+08
Moyenne (NGN),4.670645e+04,4.705301e+04
Médiane (NGN),1.030594e+04,1.010832e+04
Taux de signalement revue (%),1.960000e+01,1.924000e+01


,transactions,volume,montant_moyen,taux_signalement
Transaction_Status,,,,
Failed,630,2.482575e+07,39405.95,26.98
Pending,188,9.218044e+06,49032.15,17.55
Reversed,326,1.562612e+07,47932.88,18.40
Successful,10856,5.108074e+08,47053.01,19.24


In [3]:
import numpy as np

pd.options.display.float_format = "{:,.2f}".format

ok = tx[tx["Transaction_Status"] == "Successful"]
REF_DATE = tx["Transaction_DateTime"].max()

rfm = (
    ok.groupby("Customer_ID")
      .agg(last_txn=("Transaction_DateTime", "max"),
           frequency=("Transaction_ID", "count"),
           monetary=("Amount_NGN", "sum"))
      .reset_index()
)
rfm = customers[["Customer_ID", "Customer_Segment", "Preferred_Channel", "Account_Status"]] \
        .merge(rfm, on="Customer_ID", how="left")

rfm["frequency"] = rfm["frequency"].fillna(0).astype(int)
rfm["monetary"] = rfm["monetary"].fillna(0)
rfm["recency_days"] = (REF_DATE - rfm["last_txn"]).dt.days
rfm["recency_days"] = rfm["recency_days"].fillna(rfm["recency_days"].max() + 1)

def quartile(s, ascending=True):
    # rank(method="first") départage les égalités de façon reproductible
    return pd.qcut(s.rank(method="first", ascending=ascending), 4, labels=[1, 2, 3, 4]).astype(int)

rfm["R"] = quartile(rfm["recency_days"], ascending=False)   # 4 = plus récent
rfm["F"] = quartile(rfm["frequency"])
rfm["M"] = quartile(rfm["monetary"])

rfm["Profil"] = np.select(
    [
        rfm["frequency"] == 0,
        (rfm.R >= 3) & (rfm.F >= 3) & (rfm.M >= 3),
        (rfm.R >= 3) & (rfm.F >= 3),
        (rfm.R >= 3) & (rfm.F <= 2),
        (rfm.R <= 2) & (rfm.F >= 3) & (rfm.M >= 3),
        (rfm.R == 1) & (rfm.F <= 2),
    ],
    ["Inactifs", "Champions", "Fidèles", "Nouveaux / récents", "À risque (forte valeur)", "Perdus"],
    default="À réactiver",
)

# Synthèse par profil
summary = rfm.groupby("Profil").agg(
    clients=("Customer_ID", "count"),
    valeur=("monetary", "sum"),
    panier_moyen=("monetary", "mean"),
)
summary["% clients"] = 100 * summary["clients"] / summary["clients"].sum()
summary["% valeur"] = 100 * summary["valeur"] / summary["valeur"].sum()
display(summary.sort_values("valeur", ascending=False))

# Plages réelles de chaque score (pour interpréter les quartiles)
display(rfm.groupby("R")["recency_days"].agg(["min", "max"]))
display(rfm.groupby("F")["frequency"].agg(["min", "max"]))

# Export lisible dans Excel en français
rfm.to_csv(DATA_DIR / "rfm_python_reussies.csv", index=False, sep=";", decimal=",")

,clients,valeur,panier_moyen,% clients,% valeur
Profil,,,,,
Champions,314,"185,115,826.51","589,540.85",20.93,36.24
À risque (forte valeur),199,"109,039,169.78","547,935.53",13.27,21.35
Nouveaux / récents,283,"71,914,294.87","254,114.12",18.87,14.08
À réactiver,284,"60,613,528.34","213,427.92",18.93,11.87
Perdus,267,"57,644,374.12","215,896.53",17.80,11.28
Fidèles,153,"26,480,247.84","173,073.52",10.20,5.18


,min,max
R,,
1,17,85
2,8,17
3,3,8
4,0,3


,min,max
F,,
1,1,5
2,5,7
3,7,9
4,9,18


In [4]:
MIN_TXN = 5
Z_THRESHOLD = 2.0
NIGHT_HOURS = [23, 0, 1, 2, 3, 4]

g = ok.groupby("Customer_ID")["Amount_NGN"]
an = ok.assign(
    nb_txn=g.transform("count"),
    avg_amount=g.transform("mean"),
    std_amount=g.transform("std"),
)
an["spike_factor"] = an["Amount_NGN"] / an["avg_amount"]
an["z_score"] = (an["Amount_NGN"] - an["avg_amount"]) / an["std_amount"]

anomalies = an[(an["nb_txn"] >= MIN_TXN) & (an["z_score"] > Z_THRESHOLD)].copy()

anomalies["hour"] = anomalies["Transaction_DateTime"].dt.hour
anomalies["is_night"] = anomalies["hour"].isin(NIGHT_HOURS)
anomalies["is_international"] = to_flag(anomalies["International_Transaction"])
anomalies["flagged"] = to_flag(anomalies["Risk_Review_Flag"])

anomalies["risk_score"] = (
    anomalies["is_night"].astype(int)
    + anomalies["is_international"].astype(int)
    + (anomalies["spike_factor"] >= 8).astype(int)
)
anomalies["Risk_Category"] = pd.cut(
    anomalies["risk_score"], [-1, 0, 1, 3], labels=["Faible", "Moyen", "Élevé"]
)

print("Anomalies :", len(anomalies),
      "| clients :", anomalies["Customer_ID"].nunique(),
      "| volume :", f"{anomalies['Amount_NGN'].sum():,.0f} NGN")

display(anomalies.groupby("Risk_Category", observed=True).agg(
    transactions=("Transaction_ID", "count"),
    volume=("Amount_NGN", "sum"),
    signalees_pct=("flagged", lambda s: s.mean() * 100),
))

display(anomalies.groupby("Transaction_Type").agg(
    transactions=("Transaction_ID", "count"),
    volume=("Amount_NGN", "sum"),
    facteur_moyen=("spike_factor", "mean"),
).sort_values("volume", ascending=False))

print("Niveau de risque contre signalement existant :")
display(pd.crosstab(anomalies["Risk_Category"], anomalies["flagged"]))

cols = ["Transaction_ID", "Customer_ID", "Transaction_DateTime", "Transaction_Type",
        "Amount_NGN", "spike_factor", "is_night", "is_international", "risk_score"]
print("Anomalies non signalées à examiner en priorité :")
display(anomalies[~anomalies["flagged"]]
        .sort_values(["risk_score", "spike_factor"], ascending=False)[cols].head(10))

anomalies.to_csv(DATA_DIR / "anomalies_python_reussies.csv", index=False, sep=";", decimal=",")

Anomalies : 645 | clients : 642 | volume : 156,820,593 NGN


,transactions,volume,signalees_pct
Risk_Category,,,
Faible,456,"108,073,707.25",30.70
Moyen,182,"46,374,599.80",43.41
Élevé,7,"2,372,286.16",42.86


,transactions,volume,facteur_moyen
Transaction_Type,,,
Transfer,300,"77,608,006.38",5.07
Deposit,165,"56,624,950.38",5.61
Cash Withdrawal,84,"12,801,902.68",4.66
Card Purchase,70,"8,394,289.45",4.67
Bill Payment,19,"1,169,842.14",4.50
Airtime/Data,7,"221,602.18",3.98


Niveau de risque contre signalement existant :


flagged,False,True
Risk_Category,,
Faible,316,140
Moyen,103,79
Élevé,4,3


Anomalies non signalées à examiner en priorité :


,Transaction_ID,Customer_ID,Transaction_DateTime,Transaction_Type,Amount_NGN,spike_factor,is_night,is_international,risk_score
1210,FT-T001211,FT-C01054,2026-01-10 01:49:00,Deposit,"572,229.73",9.67,True,False,2
3882,FT-T003883,FT-C01365,2026-01-30 02:49:00,Transfer,"399,193.69",9.65,True,False,2
5083,FT-T005084,FT-C01208,2026-02-08 03:00:00,Cash Withdrawal,"117,059.05",8.59,True,False,2
9357,FT-T009358,FT-C00343,2026-03-12 04:24:00,Transfer,"148,052.03",8.34,True,False,2
2108,FT-T002109,FT-C00988,2026-01-16 19:28:00,Deposit,"643,079.27",9.87,False,False,1
9522,FT-T009523,FT-C00225,2026-03-13 10:06:00,Deposit,"464,814.64",9.69,False,False,1
10962,FT-T010963,FT-C00699,2026-03-24 05:19:00,Deposit,"449,342.03",9.45,False,False,1
2220,FT-T002221,FT-C01388,2026-01-17 15:37:00,Transfer,"331,990.39",9.25,False,False,1
11103,FT-T011104,FT-C00351,2026-03-25 06:42:00,Transfer,"318,652.28",8.72,False,False,1
570,FT-T000571,FT-C00125,2026-01-05 06:36:00,Transfer,"387,967.08",8.51,False,False,1


In [5]:
# 1) Mêmes règles que le SQL, sur toutes les transactions
g_all = tx.groupby("Customer_ID")["Amount_NGN"]
all_an = tx.assign(nb_txn=g_all.transform("count"),
                   avg_amount=g_all.transform("mean"),
                   std_amount=g_all.transform("std"))
all_an["z"] = (all_an["Amount_NGN"] - all_an["avg_amount"]) / all_an["std_amount"]
sql_like = all_an[(all_an["nb_txn"] >= MIN_TXN) & (all_an["z"] > Z_THRESHOLD)]
print("Anomalies (toutes transactions) :", len(sql_like),
      "|", f"{sql_like['Amount_NGN'].sum():,.0f} NGN")

# 2) Part de nuit et part internationale
night_all = ok["Transaction_DateTime"].dt.hour.isin(NIGHT_HOURS).mean() * 100
print(f"Part de nuit : anomalies {anomalies['is_night'].mean()*100:.1f} % "
      f"| toutes transactions réussies {night_all:.1f} %")
print(f"Part internationale (anomalies) : {anomalies['is_international'].mean()*100:.1f} %")

# 3) Profils RFM par segment et par canal
for col in ["Customer_Segment", "Preferred_Channel"]:
    print(f"\nProfils par {col} (% de clients) :")
    display(pd.crosstab(rfm[col], rfm["Profil"], normalize="index").mul(100).round(1))
    display(rfm.groupby(col).agg(clients=("Customer_ID", "count"),
                                 valeur=("monetary", "sum"),
                                 panier_moyen=("monetary", "mean")))

# 4) Statut du compte contre activité réelle
display(rfm.groupby("Account_Status").agg(clients=("Customer_ID", "count"),
                                          frequence_moyenne=("frequency", "mean"),
                                          montant_moyen=("monetary", "mean")))

# 5) Volume mensuel (transactions réussies)
display(ok.groupby(ok["Transaction_DateTime"].dt.to_period("M")).agg(
    transactions=("Transaction_ID", "count"), volume=("Amount_NGN", "sum")))

Anomalies (toutes transactions) : 750 | 184,512,474 NGN
Part de nuit : anomalies 24.2 % | toutes transactions réussies 25.2 %
Part internationale (anomalies) : 3.3 %

Profils par Customer_Segment (% de clients) :


Profil,Champions,Fidèles,Nouveaux / récents,Perdus,À risque (forte valeur),À réactiver
Customer_Segment,,,,,,
Everyday,20.30,11.40,19.70,18.80,12.70,17.20
Premium,22.40,8.50,17.60,15.90,14.20,21.40
SME,22.20,7.90,16.70,17.10,12.00,24.10
Student,20.10,10.80,19.80,17.60,14.70,16.90


,clients,valeur,panier_moyen
Customer_Segment,,,
Everyday,711,"235,721,373.49","331,534.98"
Premium,295,"101,048,806.38","342,538.33"
SME,216,"76,656,027.50","354,889.02"
Student,278,"97,381,234.09","350,292.21"



Profils par Preferred_Channel (% de clients) :


Profil,Champions,Fidèles,Nouveaux / récents,Perdus,À risque (forte valeur),À réactiver
Preferred_Channel,,,,,,
Mobile App,22.10,10.30,17.40,18.60,13.50,18.10
USSD,17.80,10.00,20.10,19.20,11.00,21.90
Web,19.90,10.10,21.80,14.80,14.00,19.30


,clients,valeur,panier_moyen
Preferred_Channel,,,
Mobile App,924,"317,784,128.01","343,922.22"
USSD,219,"70,961,263.04","324,024.03"
Web,357,"122,062,050.41","341,910.51"


,clients,frequence_moyenne,montant_moyen
Account_Status,,,
Active,1367,7.22,"339,890.90"
Dormant,107,7.36,"333,014.90"
Restricted,26,7.77,"405,537.84"


,transactions,volume
Transaction_DateTime,,
2026-01,3720,"171,270,028.52"
2026-02,3394,"160,166,567.87"
2026-03,3742,"179,370,845.07"


In [6]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

OUTPUT = DATA_DIR / "FinTrust_Resultats_Python.xlsx"

# --- Tableaux à exporter -------------------------------------------------------
stats_generales = result.reset_index().rename(columns={"index": "Indicateur"})
stats_statut = by_status.reset_index()
rfm_profils = summary.sort_values("valeur", ascending=False).reset_index()

rfm_segment_pct = (pd.crosstab(rfm["Customer_Segment"], rfm["Profil"], normalize="index")
                   .mul(100).round(1).reset_index().rename_axis(columns=None))
rfm_canal_pct = (pd.crosstab(rfm["Preferred_Channel"], rfm["Profil"], normalize="index")
                 .mul(100).round(1).reset_index().rename_axis(columns=None))

valeur_segment = rfm.groupby("Customer_Segment").agg(
    clients=("Customer_ID", "count"), valeur=("monetary", "sum"),
    panier_moyen=("monetary", "mean")).reset_index()
valeur_canal = rfm.groupby("Preferred_Channel").agg(
    clients=("Customer_ID", "count"), valeur=("monetary", "sum"),
    panier_moyen=("monetary", "mean")).reset_index()
statut_compte = rfm.groupby("Account_Status").agg(
    clients=("Customer_ID", "count"), frequence_moyenne=("frequency", "mean"),
    montant_moyen=("monetary", "mean")).reset_index()

mensuel = ok.groupby(ok["Transaction_DateTime"].dt.to_period("M")).agg(
    transactions=("Transaction_ID", "count"), volume=("Amount_NGN", "sum")).reset_index()
mensuel["Transaction_DateTime"] = mensuel["Transaction_DateTime"].astype(str)
mensuel = mensuel.rename(columns={"Transaction_DateTime": "Mois"})

plages_scores = pd.concat([
    rfm.groupby("R")["recency_days"].agg(recence_min="min", recence_max="max"),
    rfm.groupby("F")["frequency"].agg(frequence_min="min", frequence_max="max"),
], axis=1).reset_index().rename(columns={"R": "Score", "index": "Score"})

anom_risque = anomalies.groupby("Risk_Category", observed=True).agg(
    transactions=("Transaction_ID", "count"), volume=("Amount_NGN", "sum"),
    signalees_pct=("flagged", lambda s: s.mean() * 100)).reset_index()
anom_type = anomalies.groupby("Transaction_Type").agg(
    transactions=("Transaction_ID", "count"), volume=("Amount_NGN", "sum"),
    facteur_moyen=("spike_factor", "mean")).sort_values("volume", ascending=False).reset_index()
anom_signalement = (pd.crosstab(anomalies["Risk_Category"], anomalies["flagged"])
                    .rename(columns={False: "Non signalée", True: "Signalée"})
                    .reset_index().rename_axis(columns=None))

cols_prio = ["Transaction_ID", "Customer_ID", "Transaction_DateTime", "Transaction_Type",
             "Amount_NGN", "spike_factor", "is_night", "is_international", "risk_score"]
anom_prioritaires = (anomalies[~anomalies["flagged"]]
                     .sort_values(["risk_score", "spike_factor"], ascending=False)[cols_prio]
                     .head(20))

parametres = pd.DataFrame({
    "Paramètre": ["Transactions analysées", "Période", "Date de référence (récence)",
                  "Scores RFM", "Seuil d'anomalie", "Historique minimal", "Heures de nuit"],
    "Valeur": ["Statut Successful uniquement",
               f"{tx['Transaction_DateTime'].min():%d/%m/%Y} au {tx['Transaction_DateTime'].max():%d/%m/%Y}",
               f"{REF_DATE:%d/%m/%Y}", "Quartiles 1 à 4 (4 = meilleur)",
               f"Moyenne du client + {Z_THRESHOLD} écarts-types",
               f"{MIN_TXN} transactions", "23 h à 4 h 59"],
})

sheets = {
    "Parametres": parametres,
    "Stats generales": stats_generales,
    "Stats par statut": stats_statut,
    "Activite mensuelle": mensuel,
    "RFM profils": rfm_profils,
    "RFM plages scores": plages_scores,
    "RFM segment %": rfm_segment_pct,
    "RFM canal %": rfm_canal_pct,
    "Valeur par segment": valeur_segment,
    "Valeur par canal": valeur_canal,
    "Statut compte": statut_compte,
    "Anomalies risque": anom_risque,
    "Anomalies type": anom_type,
    "Anomalies signalement": anom_signalement,
    "Anomalies prioritaires": anom_prioritaires,
    "RFM clients (detail)": rfm,
    "Anomalies (detail)": anomalies,
}

# --- Écriture et mise en forme ---------------------------------------------------
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")

with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name[:31], index=False)
        ws = writer.sheets[name[:31]]

        for cell in ws[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = HEADER_FILL
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.freeze_panes = "A2"

        for i, col in enumerate(df.columns, start=1):
            longest = df[col].astype(str).str.len().max() if len(df) else 0
            ws.column_dimensions[get_column_letter(i)].width = min(max(len(str(col)), longest) + 2, 40)

            if pd.api.types.is_float_dtype(df[col]):
                fmt = "#,##0.00"
            elif pd.api.types.is_integer_dtype(df[col]):
                fmt = "#,##0"
            elif pd.api.types.is_datetime64_any_dtype(df[col]):
                fmt = "dd/mm/yyyy hh:mm"
            else:
                continue
            for row in ws.iter_rows(min_row=2, min_col=i, max_col=i):
                for c in row:
                    c.number_format = fmt

print("Fichier créé :", OUTPUT)
print(len(sheets), "feuilles")

Fichier créé : F:\FunTrust\WEEK 4\FinTrust_Resultats_Python.xlsx
17 feuilles
